In [35]:
import os
import pandas as pd
import numpy as np 
import seaborn as sns
from tqdm import tqdm
import pickle

from pydub import AudioSegment
from IPython.display import Audio

import librosa
import librosa.display
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, accuracy_score
import itertools

import warnings
warnings.filterwarnings("ignore")

In [36]:
ds_train = pd.read_csv('/Users/camcortes/Documents/birds-sounds/data/chunks_specie_sample_train.csv')
ds_test = pd.read_csv('/Users/camcortes/Documents/birds-sounds/data/chunks_specie_sample_test.csv')
ds_train = ds_train[['audio_path', 'label']]
ds_test = ds_test[['audio_path', 'label']]
print(ds_train.shape)
print(ds_test.shape)
ds_train.sample(5)

(19927, 2)
(8722, 2)


,audio_path,label
2692,chunks/106958_chunk31.mp3,Icterus galbula
2550,chunks/234353_chunk21.mp3,Catharus aurantiirostris
5950,chunks/340079_chunk34.mp3,Myiarchus swainsoni
1348,chunks/289623_chunk22.mp3,Chamaeza campanisona
11255,chunks/3179_chunk9.mp3,Mionectes oleagineus


In [37]:
# label encoder
le = LabelEncoder()
ds_train['label'] = le.fit_transform(ds_train['label'])
ds_test['label'] = le.transform(ds_test['label'])

In [38]:
etiquetas = ds_train['label']
clases, conteo_clases = np.unique(etiquetas, return_counts=True)
weight = dict(zip(clases, np.max(conteo_clases) / conteo_clases))

In [39]:
X = ds_train['audio_path']
y = ds_train['label']

X_test = ds_test['audio_path']
y_test = ds_test['label']

In [40]:
num_clases = y.nunique()
print(num_clases)

133


In [41]:
#getting mfcc features for all the data points in test and train

def get_spec(path):
    y, sr = librosa.load(path, sr=None, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=10000)
    return S

In [42]:
def process_audio_files(X, y):
    """
    Processes a list of audio file paths to extract MFCC features and resize them.

    Args:
        X (pd.Series): A pandas Series containing paths to audio files.
        y (pd.Series): A pandas Series containing labels corresponding to the audio files.

    Returns:
        tuple: A tuple containing:
            - X_processed (np.ndarray): A numpy array of processed MFCC features with shape (num_files, 65, 40).
            - y_processed (np.ndarray): A numpy array of labels corresponding to the audio files.
    """
    temp = []
    label = []
    for i in tqdm(range(len(X))):
        audio_path = X.iloc[i]
        try:
            spec = get_spec(audio_path)
            spec = np.resize(spec, (65, 40,))
            temp.append(spec)
            label.append(y.iloc[i])
        except FileNotFoundError as e:
            print(e)
        except Exception as e:
            print(f"Error procesando el archivo {audio_path}: {e}")
    X_processed = np.asarray(temp)
    y_processed = np.asarray(label)
    return X_processed, y_processed

In [ ]:
#%%time
X_processed, y = process_audio_files(X, y)
X_test_processed, y_test = process_audio_files(X_test, y_test)

  2%|▏         | 303/19927 [00:03<02:16, 143.61it/s]

[Errno 2] No such file or directory: 'chunks/281324_chunk16.mp3'


  2%|▏         | 318/19927 [00:03<02:34, 127.04it/s]

In [ ]:
# with open('X_processed_spec.pkl', 'wb') as f:
#     pickle.dump(X_processed, f)
    
# with open('y_train_spec.pkl', 'wb') as f:
#     pickle.dump(y, f)

# with open('X_test_processed_spec.pkl', 'wb') as f:
#     pickle.dump(X_test_processed, f)
    
# with open('y_test_spec.pkl', 'wb') as f:
#     pickle.dump(y_test, f)

In [ ]:
with open('X_processed_spec.pkl', 'rb') as f:
    X_processed = pickle.load(f)

with open('y_train_spec.pkl', 'rb') as f:
    y = pickle.load(f)

with open('X_test_processed_spec.pkl', 'rb') as f:
    X_test_processed = pickle.load(f)
    
with open('y_test_spec.pkl', 'rb') as f:
    y_test = pickle.load(f)

In [ ]:
print(f'El conjunto de entrenamiento tiene la forma {X_processed.shape}')
print(f'Los labels del conjunto de entrenamiento tiene la forma {y.shape}')

El conjunto de entrenamiento tiene la forma (19926, 65, 40)
Los labels del conjunto de entrenamiento tiene la forma (19926,)


#### Modelling

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
import keras.backend as K

tf.keras.backend.clear_session()

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [ ]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Add attention layer to the deep learning network
class Attention(layers.Layer):
    """
    Custom Keras Layer implementing an Attention mechanism.

    Methods
    -------
    build(input_shape)
        Initializes the weights and biases for the attention mechanism.
    
    call(x)
        Applies the attention mechanism to the input tensor `x` and returns the context vector.

    Parameters
    ----------
    input_shape : tuple
        Shape of the input tensor.
    
    x : tensor
        Input tensor to which the attention mechanism is applied.

    Returns
    -------
    tensor
        Context vector obtained after applying the attention mechanism.
    """
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], 1), 
                                 initializer='random_normal', trainable=True)
        self.b = self.add_weight(name='attention_bias', shape=(input_shape[1], 1), 
                                 initializer='zeros', trainable=True)
        super(Attention, self).build(input_shape)

    def call(self, x):
        # Alignment scores. Pass them through tanh function
        e = tf.tanh(tf.matmul(x, self.W) + self.b)  # Cambiado K.dot por tf.matmul
        # Remove dimension of size 1
        e = tf.squeeze(e, axis=-1)  # Cambiado K.squeeze por tf.squeeze
        # Compute the weights
        alpha = tf.nn.softmax(e)  # Cambiado K.softmax por tf.nn.softmax
        # Reshape to tensorFlow format
        alpha = tf.expand_dims(alpha, axis=-1)  # Cambiado K.expand_dims por tf.expand_dims
        # Compute the context vector
        context = x * alpha
        context = tf.reduce_sum(context, axis=1)  # Cambiado K.sum por tf.reduce_sum
        return context

In [ ]:
#Create model

#input
input = layers.Input(shape=(65,40), name='input_layer')

#cnn
x = layers.Conv1D(256, (3,), activation='relu')(input)
x = layers.BatchNormalization()(x)
x = layers.Conv1D(128, (3,), activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv1D(64, (3,), activation='relu')(x)
x = layers.BatchNormalization()(x)
#lstm
x = layers.LSTM(256, return_sequences = True)(x)
x = layers.Dropout(0.5)(x)
x = layers.LSTM(128, return_sequences = True)(x)
x = layers.Dropout(0.5)(x)
x = layers.LSTM(64, return_sequences = True)(x)
x = layers.Dropout(0.5)(x)
x = Attention()(x)
#dense
x = layers.Dense(256,'relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128,'relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(64,'relu')(x)
x = layers.Dropout(0.5)(x)

#output
output = layers.Dense(num_clases, activation='softmax', name='output_layer')(x)

model = Model(inputs=input, outputs=output)
model.summary()

2024-11-24 00:12:34.385122: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2024-11-24 00:12:34.385153: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2024-11-24 00:12:34.385160: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2024-11-24 00:12:34.385511: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2024-11-24 00:12:34.385528: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 65, 40)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 63, 256)        │        30,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 63, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 61, 128)        │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 59, 64)         │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 59, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 59, 256)        │       328,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 59, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 59, 128)        │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 59, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 59, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 59, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention (Attention)           │ (None, 64)             │           123 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 133)            │         8,645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 797,632 (3.04 MB)

 Trainable params: 796,736 (3.04 MB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    'model_spec_species.keras',  
    monitor='val_loss',
    mode='min',
    verbose=1,
    save_weights_only=False,  # guardar solo los pesos
    save_best_only=True,     # guardar solo el mejor modelo
)

# Callback para TensorBoard
tensorboard_cb = tf.keras.callbacks.TensorBoard(
    log_dir='logs',
    histogram_freq=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.1, 
    patience=10, 
    min_lr=0.00001)

# Callback para EarlyStopping
early_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=50,
    mode='min'
)

callbacks = [checkpoint_cb, tensorboard_cb, early_callback, reduce_lr]

In [ ]:
#compile
ls = tf.keras.losses.SparseCategoricalCrossentropy()
adam = tf.keras.optimizers.Adam()
model.compile(adam, ls, metrics=['accuracy'])

#fitting
sss = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=42)

for train_index, val_index in sss.split(np.zeros(X_processed.shape[0]), y):
    X_train, X_val = X_processed[train_index], X_processed[val_index]
    y_train_split, y_val = y[train_index], y[val_index] 

    # Entrenamiento del modelo
    history = model.fit(
        X_train, y_train_split, 
        validation_data=(X_val, y_val),
        epochs=400,
        callbacks=callbacks,
        class_weight=weight,
        batch_size=256)

Epoch 1/400
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 243ms/step - accuracy: 0.0259 - loss: 31.5314
Epoch 1: val_loss did not improve from 5.23064
55/55 ━━━━━━━━━━━━━━━━━━━━ 22s 273ms/step - accuracy: 0.0259 - loss: 31.5286 - val_accuracy: 0.0309 - val_loss: 5.2353 - learning_rate: 0.0010
Epoch 2/400
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 227ms/step - accuracy: 0.0283 - loss: 31.1490
Epoch 2: val_loss improved from 5.23064 to 5.19777, saving model to model_spec_species.keras
55/55 ━━━━━━━━━━━━━━━━━━━━ 14s 243ms/step - accuracy: 0.0283 - loss: 31.1501 - val_accuracy: 0.0316 - val_loss: 5.1978 - learning_rate: 0.0010
Epoch 3/400
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 225ms/step - accuracy: 0.0291 - loss: 30.9802
Epoch 3: val_loss improved from 5.19777 to 4.77633, saving model to model_spec_species.keras
55/55 ━━━━━━━━━━━━━━━━━━━━ 13s 241ms/step - accuracy: 0.0291 - loss: 30.9830 - val_accuracy: 0.0325 - val_loss: 4.7763 - learning_rate: 0.0010
Epoch 4/400
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step - accuracy: 0.0249 - 

KeyboardInterrupt: 

In [ ]:
fig = plt.figure(figsize=(12, 5))
fig.add_subplot(121)

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model: loss vs epochs')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Training', 'Validation'], loc='upper right')

fig.add_subplot(122)

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model: acc vs epochs')
plt.ylabel('Acc')
plt.xlabel('Epoch')
plt.legend(['Training', 'Validation'], loc='lower right')
plt.show()

#### Predictions

In [ ]:
# model.load_weights('model_weights_spec.weights.h5')
# model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
preds = model.predict(X_test_processed)
model.evaluate(X_test_processed, y_test)

In [ ]:
cm = confusion_matrix(y_test, np.argmax(preds, axis=1))

# Número de clases
num_classes = cm.shape[0]

# Normalizar la matriz de confusión
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Graficar
plt.figure(figsize=(8, 6))  # Opcional, para ajustar el tamaño de la figura
plt.imshow(cm_normalized, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()

tick_marks = np.arange(num_classes)
plt.xticks(tick_marks, range(num_classes), rotation=45)
plt.yticks(tick_marks, range(num_classes))

# Anotar la matriz
for i, j in itertools.product(range(cm_normalized.shape[0]), range(cm_normalized.shape[1])):
    plt.text(j, i, f"{cm_normalized[i, j]:.2%}",
             horizontalalignment="center",
             color="white" if cm_normalized[i, j] > 0.5 else "black")

plt.tight_layout()
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
roc_auc_score(y_test, np.argmax(preds, axis=1))

In [ ]:
accuracy_score(y_test, np.argmax(preds, axis=1))

In [ ]:
print(classification_report(y_test, np.argmax(preds, axis=1)))